# Generate All Evidence for Project Write-Up

Run this notebook in **Google Colab** (Runtime → Run all) to generate every table, figure, and metric needed for your appendices.

**Setup time:** ~5 min  |  **Run time:** ~15 min (with T4 GPU)

---

In [ ]:
# @title 1. Mount Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

# Install any missing packages
!pip install -q torch transformers[torch] shap streamlit matplotlib seaborn scikit-learn pandas numpy

import os, sys, json, time, re, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve, average_precision_score,
                             classification_report)

import torch
from torch.optim import AdamW
from transformers import (DistilBertForSequenceClassification,
                          DistilBertTokenizerFast, Trainer, TrainingArguments,
                          EarlyStoppingCallback)
import shap

warnings.filterwarnings('ignore')
print('Setup complete. PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# @title 2. Upload Project Files to Colab
# Run this cell, then upload your genai-phishing-detector ZIP
from google.colab import files
print('Please upload your genai-phishing-detector.zip file')
uploaded = files.upload()

import zipfile
with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
    z.extractall('/content/')

os.chdir('/content/genai-phishing-detector')
print('Files extracted. Contents:', os.listdir('.'))

In [ ]:
# @title 3. Appendix 13: Dataset Composition Table
df = pd.read_csv('dataset/dataset.csv')
label_map = {0: 'Legitimate Financial Communication',
             1: 'Traditional Phishing',
             2: 'AI-Generated Phishing'}
df['class_name'] = df['label'].map(label_map)
counts = df['label'].value_counts().sort_index()

print('='*60)
print('Table 4.1: Dataset Composition')
print('='*60)
print(f'{"Class":40s} {"Samples":>8s}')
print('-'*60)
for i in range(3):
    print(f'{label_map[i]:40s} {counts[i]:>8d}')
print('-'*60)
print(f'{"Total":40s} {len(df):>8d}')
print('='*60)

In [ ]:
# @title 4. Appendix 12: Dataset Distribution Bar Chart
colors = ['#2ecc71', '#f39c12', '#e74c3c']
labels = ['Legitimate\nFinancial Comm.', 'Traditional\nPhishing', 'AI-Generated\nPhishing']

plt.figure(figsize=(9, 5))
bars = plt.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
             str(count), ha='center', va='bottom', fontweight='bold', fontsize=13)
plt.ylabel('Number of Samples', fontsize=12)
plt.title('Figure 4.1: Dataset Class Distribution', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/dataset_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/dataset_distribution.png')

In [ ]:
# @title 5. Load Trained Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model_path = 'models/phishing_model'
tokenizer_path = 'models/tokenizer'

# Extract ZIP if needed
if os.path.exists('models/phishing_model.zip'):
    with zipfile.ZipFile('models/phishing_model.zip', 'r') as z:
        z.extractall('models/')
if os.path.exists('models/tokenizer.zip'):
    with zipfile.ZipFile('models/tokenizer.zip', 'r') as z:
        z.extractall('models/')

tokenizer = DistilBertTokenizerFast.from_pretrained(tokenizer_path)
model = DistilBertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()
print('Model loaded successfully')

In [ ]:
# @title 6. Appendix 5b: Confusion Matrix & Appendix 5a: Model Performance
os.makedirs('figures', exist_ok=True)

# Prepare test set
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_size = int(0.8 * len(df))
val_size = int(0.1 * len(df))
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:train_size+val_size]
test_df = df.iloc[train_size+val_size:]

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

# Evaluate on test set
texts = test_df['text'].tolist()
y_true = test_df['label'].values

all_preds = []
all_probs = []
batch_size = 32
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    enc = tokenizer(batch_texts, truncation=True, padding=True,
                    max_length=128, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        outputs = model(**enc)
    probs = torch.softmax(outputs.logits, dim=-1)
    preds = torch.argmax(probs, dim=-1)
    all_preds.extend(preds.cpu().tolist())
    all_probs.extend(probs.cpu().numpy())

y_pred = np.array(all_preds)
y_prob = np.array(all_probs)

# Metrics
acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted', zero_division=0)

print()
print('='*50)
print('Table 4.4: Model Performance')
print('='*50)
print(f'{"Metric":20s} {"Result":>10s}')
print('-'*50)
print(f'{"Accuracy":20s} {acc:>10.4f}')
print(f'{"Precision (weighted)":20s} {precision:>10.4f}')
print(f'{"Recall (weighted)":20s} {recall:>10.4f}')
print(f'{"F1-Score (weighted)":20s} {f1:>10.4f}')

# AUPRC (one-vs-rest macro)
auprc_scores = []
for i in range(3):
    y_true_bin = (y_true == i).astype(int)
    score = average_precision_score(y_true_bin, y_prob[:, i])
    auprc_scores.append(score)
auprc_macro = np.mean(auprc_scores)
print(f'{"AUPRC (macro)":20s} {auprc_macro:>10.4f}')
print('='*50)

# Per-class metrics
print()
print(classification_report(y_true, y_pred,
      target_names=['Legitimate', 'Trad Phishing', 'AI Phishing']))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
    display_labels=['Legitimate', 'Trad Phishing', 'AI Phishing'])
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix — DistilBERT Phishing Detector', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/confusion_matrix.png')

# --- Text Confusion Matrix (verifiable without image) ---
label_w = max(len(l) for l in ['Legitimate', 'Trad Phishing', 'AI Phishing'])
cell_w = max(len(str(cm.max())), 6) + 2
col_hdrs = ['P:Legitimate', 'P:Trad Phish', 'P:AI Phish']
row_hdrs = ['T:Legitimate', 'T:Trad Phish', 'T:AI Phish']

sep = '+' + '-'*(label_w+2) + '+' + '+'.join('-'*cell_w for _ in range(3)) + '+'
print()
print('Confusion Matrix (text):')
print(sep)
print('|' + ' '*(label_w+2) + '|' + '|'.join(f'{h:^{cell_w}}' for h in col_hdrs) + '|')
print(sep.replace('-', '='))
for i in range(3):
    print(f'|{row_hdrs[i]:>{label_w+2}}|' + '|'.join(f'{cm[i][j]:^{cell_w}}' for j in range(3)) + '|')
print(sep)

off_diag = int(cm.sum() - np.trace(cm))
total = int(cm.sum())
print(f'\nOverall accuracy: {int(np.trace(cm))}/{total} = {np.trace(cm)/total:.4f}')
if off_diag == 0:
    print('CONFIRMED: Zero misclassifications across all classes.')
else:
    print(f'{off_diag} misclassification(s) found.')

print('\n' + '='*60)
print('CAVEAT: Perfect 1.0000 test-set scores should not be interpreted')
print('as evidence of guaranteed real-world performance. The dataset is')
print('procedurally generated from a small number of templates with minor')
print('slot substitutions. Only exact-string deduplication was applied,')
print('so near-duplicate or template-sibling samples may span both the')
print('train and test splits. Run validation/data_integrity.py --fast')
print('to check for near-duplicate contamination before submission.')
print('='*60)

In [ ]:
# @title 7. Appendix 3: Inference Latency Table
times_ms = []
sample_texts = test_df['text'].sample(100, random_state=42).tolist()

for text in sample_texts:
    enc = tokenizer(text, truncation=True, padding='max_length',
                    max_length=128, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(**enc)
    elapsed = (time.perf_counter() - start) * 1000
    times_ms.append(elapsed)

times = np.array(times_ms)
print()
print('='*50)
print('Table 4.6: Inference Latency')
print('='*50)
print(f'{"Metric":25s} {"Value":>15s}')
print('-'*50)
print(f'{"Average Response Time":25s} {times.mean():>10.2f} ms')
print(f'{"Fastest Response":25s} {times.min():>10.2f} ms')
print(f'{"Slowest Response":25s} {times.max():>10.2f} ms')
print(f'{"Std Deviation":25s} {times.std():>10.2f} ms')
print(f'{"Samples Tested":25s} {len(times):>10d}')
print('='*50)

In [ ]:
# @title 8. Appendix 4: Regex Baseline Comparison
def regex_classify(text):
    t = text.lower()
    # AI phishing indicators
    ai_score = 0
    ai_patterns = ['central bank of nigeria', 'cbn directive', 'regulatory compliance',
                   'beneficial ownership', 'anti-money laundering', 'know your customer',
                   'aml', 'kyc', 'compliance notice', 'mandatory']
    for p in ai_patterns:
        if p in t: ai_score += 1

    # Traditional phishing indicators
    trad_score = 0
    trad_patterns = ['urgent', 'click here', 'suspended', 'blocked',
                     'verify.*account', 'bvn.*block', 'reactivate',
                     'immediate', 'warning', 'expired']
    for p in trad_patterns:
        if re.search(p, t): trad_score += 1

    # Legitimate indicators
    legit_score = 0
    legit_patterns = ['debit alert', 'credit alert', 'available balance',
                      'transaction alert', 'monthly statement', 'account balance',
                      'deposit', 'withdrawal']
    for p in legit_patterns:
        if p in t: legit_score += 1

    if ai_score > trad_score and ai_score > legit_score:
        return 2
    elif trad_score > legit_score:
        return 1
    return 0

y_pred_regex = np.array([regex_classify(t) for t in texts])

acc_regex = accuracy_score(y_true, y_pred_regex)
p_regex, r_regex, f_regex, _ = precision_recall_fscore_support(
    y_true, y_pred_regex, average='weighted', zero_division=0)

acc_model = accuracy_score(y_true, y_pred)
p_model, r_model, f_model, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted', zero_division=0)

print()
print('='*65)
print('Table 4.5: Comparison with Regex Baseline')
print('='*65)
print(f'{"Metric":12s} {"Regex-Based":>16s} {"Proposed":>16s} {"Improvement":>14s}')
print('-'*65)
print(f'{"Accuracy":12s} {acc_regex:>10.4f}      {acc_model:>10.4f}      {(acc_model-acc_regex):>+8.4f}')
print(f'{"Precision":12s} {p_regex:>10.4f}      {p_model:>10.4f}      {(p_model-p_regex):>+8.4f}')
print(f'{"Recall":12s} {r_regex:>10.4f}      {r_model:>10.4f}      {(r_model-r_regex):>+8.4f}')
print(f'{"F1-Score":12s} {f_regex:>10.4f}      {f_model:>10.4f}      {(f_model-f_regex):>+8.4f}')
print('='*65)

In [ ]:
# @title 9. Appendix 2: Zero-Shot Detection Results
scenarios = [
    ('Scenario 1: Legitimate Transaction',
     'Dear Chidi, a debit of NGN25,000 was made on your GTBank account '
     '0123456789 at ShopRite on 15-Jan-2025. Available balance: NGN450,000.'),
    ('Scenario 2: Traditional Phishing',
     'URGENT!!! Your BVN has been BLOCKED! Click here to verify now: '
     'https://account-verify.tk/38472'),
    ('Scenario 3: AI-Generated Phishing',
     'Subject: Mandatory BVN-NIN Linkage Compliance Notice\n\n'
     'Dear Customer, the Central Bank of Nigeria (CBN) requires all accounts '
     'to have BVN linked to NIN by 30-Mar-2025. Non-compliant accounts will '
     'be restricted. Complete linkage: https://secure-gtbank-portal.com/verify'),
]

expected_labels = ['Legitimate Financial Comm.', 'Traditional Phishing', 'AI-Generated Phishing']
predicted_labels = ['Legitimate Financial Comm.', 'Traditional Phishing', 'AI-Generated Phishing']

print()
print('='*90)
print('Table 4.7: Zero-Shot Detection Results')
print('='*90)
print(f'{"Test Scenario":30s} {"Expected":25s} {"Predicted":25s} {"Status":>8s}')
print('-'*90)

for i, (name, text) in enumerate(scenarios):
    enc = tokenizer(text, truncation=True, padding='max_length',
                    max_length=128, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        outputs = model(**enc)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    status = 'Correct' if pred == i else 'Incorrect'
    print(f'{name:30s} {expected_labels[i]:25s} {predicted_labels[pred]:25s} {status:>8s}')
print('='*90)

In [ ]:
# @title 10. Appendix 1: SHAP Explanation Screenshot — Generate Text for Streamlit
# SHAP on Colab GPU is fast. We generate the explanation and show the output.
os.makedirs('figures', exist_ok=True)

test_text = (
    'Subject: Mandatory BVN-NIN Linkage Compliance Notice\n\n'
    'Dear Chidi Okonkwo,\n\n'
    'This is to notify you that the Central Bank of Nigeria (CBN) now requires '
    'all bank accounts to have their BVN linked to the National Identification '
    'Number (NIN) by 30-Mar-2025. Accounts not complying will be placed on '
    'restricted status.\n\n'
    'To complete the linkage securely, please visit: '
    'https://secure.gtbank-portal.com/verify-3847\n\n'
    'Thank you for your cooperation.\n'
    'Compliance Department\n'
    'GTBank'
)

# Encode
enc = tokenizer(test_text, truncation=True, padding='max_length',
                max_length=128, return_tensors='pt')
enc = {k: v.to(device) for k, v in enc.items()}

# Get prediction
with torch.no_grad():
    outputs = model(**enc)
pred_class = torch.argmax(outputs.logits, dim=-1).item()
probs = torch.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()

label_names = ['Legitimate', 'Traditional Phishing', 'AI-Generated Phishing']
print(f'Prediction: {label_names[pred_class]}')
print(f'Confidence: {probs[pred_class]:.4f}')
for i in range(3):
    print(f'  P({label_names[i]:25s}) = {probs[i]:.4f}')

# SHAP explanation (using Partition explainer)
from transformers import DistilBertForSequenceClassification as DBert

def predict_proba_batch(texts):
    if isinstance(texts, str):
        texts = [texts]
    elif isinstance(texts, np.ndarray):
        texts = texts.tolist()
    texts = list(texts)
    texts = [t if isinstance(t, str) else ' '.join(t) for t in texts]
    enc = tokenizer(texts, truncation=True, padding='max_length',
                    max_length=128, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

masker = shap.maskers.Text(tokenizer, mask_token='...', collapse_mask_token=True)
explainer = shap.Explainer(
    predict_proba_batch,
    masker,
    output_names=label_names,
    seed=42
)

print('\nRunning SHAP explanation (may take 30-60s on GPU)...')
shap_values = explainer([test_text], max_evals=50, batch_size=1)

# Generate waterfall plot
for class_idx in range(3):
    plt.figure()
    shap.waterfall_plot(
        shap_values[0, :, class_idx],
        show=False,
        max_display=15
    )
    plt.title(f'SHAP Waterfall — Class: {label_names[class_idx]}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'figures/shap_waterfall_class_{class_idx}.png',
                dpi=200, bbox_inches='tight')
    plt.show()

# Generate text explanation
words = tokenizer.convert_ids_to_tokens(
    tokenizer(test_text, truncation=True, padding='max_length',
              max_length=128)['input_ids']
)
vals = shap_values[0, :, pred_class].values
top_indices = np.argsort(np.abs(vals))[-10:]

print(f'\nTop 10 influential words (for {label_names[pred_class]}):')
print(f'{"Word":20s} {"SHAP Value":>10s} {"Impact"}')
print('-'*45)
for idx in reversed(top_indices):
    word = words[idx]
    if word in ['[PAD]', '[CLS]', '[SEP]']:
        continue
    val = vals[idx]
    impact = 'AI Phish' if val > 0.1 else ('Trad Phish' if val > 0.05 else ('Legit' if val < -0.05 else 'Neutral'))
    print(f'{word:20s} {val:>10.4f}  {impact}')

print('\nSaved SHAP waterfall plots to figures/')

In [ ]:
# @title 11. Appendix 7: Training Loss & Validation Accuracy Curves
# PASTE your actual training history here (copied from Colab training output)
# Example format — REPLACE with your actual values:
training_history = {
    'epoch': [1, 2, 3, 4, 5],
    'loss': [0.8234, 0.4532, 0.2876, 0.1987, 0.1456],        # <<< REPLACE
    'val_accuracy': [0.7215, 0.8651, 0.9213, 0.9438, 0.9562], # <<< REPLACE
    'val_f1': [0.7198, 0.8637, 0.9205, 0.9429, 0.9556],       # <<< REPLACE
}

epochs = training_history['epoch']

# Figure 4.3: Training Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs, training_history['loss'], 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Figure 4.3: Training Loss Curve', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.xticks(epochs)
plt.tight_layout()
plt.savefig('figures/training_loss_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/training_loss_curve.png')

# Figure 4.4: Validation Accuracy & F1 Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs, training_history['val_accuracy'], 'g-s', linewidth=2,
         markersize=8, label='Validation Accuracy')
plt.plot(epochs, training_history['val_f1'], 'm-d', linewidth=2,
         markersize=8, label='Validation F1 Score')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Figure 4.4: Validation Accuracy & F1 Curve', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.xticks(epochs)
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('figures/validation_accuracy_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/validation_accuracy_curve.png')

In [ ]:
# @title 12. Appendix 8: Hyperparameters Table
import sys
sys.path.insert(0, '.')
try:
    from config.config import EPOCHS, BATCH_SIZE, LEARNING_RATE, MODEL_NAME, MAX_LENGTH
except:
    EPOCHS = 5; BATCH_SIZE = 16; LEARNING_RATE = 2e-5; MODEL_NAME = 'distilbert-base-uncased'; MAX_LENGTH = 128

print()
print('='*50)
print('Table 4.3: Hyperparameter Configuration')
print('='*50)
print(f'{"Parameter":22s} {"Value":>25s}')
print('-'*50)
print(f'{"Pre-trained Model":22s} {MODEL_NAME:>25s}')
print(f'{"Max Sequence Length":22s} {MAX_LENGTH:>25d}')
print(f'{"Learning Rate":22s} {LEARNING_RATE:>25.0e}')
print(f'{"Batch Size":22s} {BATCH_SIZE:>25d}')
print(f'{"Epochs":22s} {EPOCHS:>25d}')
print(f'{"Optimizer":22s} {"AdamW":>25s}')
print(f'{"Loss Function":22s} {"Cross Entropy":>25s}')
print(f'{"Scheduler":22s} {"Linear (no warmup)":>25s}')
print(f'{"Weight Decay":22s} {"0.01":>25s}')
print('='*50)

In [ ]:
# @title 13. Download All Generated Figures
from google.colab import files
import zipfile

with zipfile.ZipFile('evidence_figures.zip', 'w') as z:
    for f in Path('figures').glob('*.png'):
        z.write(f, arcname=f.name)
        print(f'Added: {f.name}')

files.download('evidence_figures.zip')
print('\nAll figures downloaded as evidence_figures.zip')
print('\nOpen each PNG, crop as needed, and insert into your project write-up.')